# 01. Data Ingestion

This notebook handles:
1. Loading raw AEMO NEM data
2. Filtering for NSW1 region
3. Timezone conversion (Australia/Sydney)
4. Resampling to 5-minute grid
5. Saving processed data

## Data Sources
- AEMO DISPATCHPRICE: 5-minute dispatch prices
- AEMO DISPATCH_UNIT_SCADA: Unit-level generation (optional)
- OpenNEM API: Aggregated solar/wind (alternative)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
import logging

from src import config
from src import io as data_io

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Check Raw Data Files

Place your raw AEMO CSV files in `data/raw/`. Expected file formats:
- `DISPATCHPRICE_*.csv` - Price and demand data
- `DISPATCH_UNIT_SCADA_*.csv` - Generation data (optional)

In [ ]:
# List available raw files
raw_files = list(config.DATA_RAW.glob('*.csv'))
print(f"Found {len(raw_files)} CSV files in {config.DATA_RAW}:")
for f in raw_files:
    print(f"  - {f.name}")

## 2. Load and Preview Price Data

In [ ]:
# Load price data (adjust pattern to match your files)
price_pattern = str(config.DATA_RAW / '*DISPATCHPRICE*.csv')

# For single file:
# price_df = data_io.load_raw_price_data('path/to/file.csv')

# For multiple files:
try:
    price_df = data_io.load_all_raw_files(
        pattern=price_pattern,
        loader_func=data_io.load_raw_price_data,
        region=config.REGION
    )
    print(f"Loaded price data: {len(price_df)} rows")
    print(f"Date range: {price_df.index.min()} to {price_df.index.max()}")
    display(price_df.head())
except FileNotFoundError as e:
    print(f"No price files found. Please add AEMO DISPATCHPRICE CSV files to {config.DATA_RAW}")
    print("\nYou can download data from: https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/data-nem/market-data-nemweb")

## 3. Load Generation Data (Optional)

If you have solar/wind generation data, load it here.

In [ ]:
# Load generation data (if available)
gen_pattern = str(config.DATA_RAW / '*SCADA*.csv')

try:
    gen_df = data_io.load_all_raw_files(
        pattern=gen_pattern,
        loader_func=data_io.load_raw_generation_data,
        region=config.REGION
    )
    print(f"Loaded generation data: {len(gen_df)} rows")
    display(gen_df.head())
except FileNotFoundError:
    gen_df = None
    print("No generation data found. Proceeding with price/demand only.")

## 4. Resample to 5-Minute Grid

In [ ]:
# Resample price data
if 'price_df' in dir() and price_df is not None:
    price_df_resampled = data_io.resample_to_5min_grid(price_df)
    
    print(f"Resampled data: {len(price_df_resampled)} intervals")
    print(f"Missing intervals: {price_df_resampled['is_missing'].sum()}")
    display(price_df_resampled.head(10))

## 5. Merge Price and Generation Data

In [ ]:
# Merge if generation data exists
if 'price_df_resampled' in dir() and 'gen_df' in dir() and gen_df is not None:
    combined_df = data_io.merge_price_and_generation(price_df_resampled, gen_df)
else:
    combined_df = price_df_resampled if 'price_df_resampled' in dir() else None

if combined_df is not None:
    print(f"Combined data columns: {list(combined_df.columns)}")
    display(combined_df.describe())

## 6. Save Processed Data

In [ ]:
# Save to parquet
if combined_df is not None:
    output_path = data_io.save_processed_data(combined_df, 'nsw_processed')
    print(f"Saved processed data to: {output_path}")
else:
    print("No data to save. Please ensure raw data files are in place.")

## 7. Quick Validation

In [ ]:
# Verify saved data
try:
    loaded = data_io.load_processed_data('nsw_processed')
    print(f"Verification successful!")
    print(f"Shape: {loaded.shape}")
    print(f"Date range: {loaded.index.min()} to {loaded.index.max()}")
    print(f"Columns: {list(loaded.columns)}")
except FileNotFoundError:
    print("Processed data not found.")

---
## Next Steps
Proceed to `02_eda.ipynb` for exploratory data analysis.